# S5.M1-1 方法筛选：运行核对、数据与验证集上的选择

**目的**：所有者 2026-11-02 要求盘点论文 / 期刊中的优化方法在全部数据集上哪些有增益、哪些可组合，并立即补做未做的实验（[DR-M5-012](../../../registry/decision_records/DR-M5-012.yaml)）。本 notebook 核对运行来源、列出 14 个任务的数据规模，并展示只在验证集上做出的选择。

**引擎**：明文直方图 GBDT（XGBoost hist）代理。隐语 SGB 与同分桶、同超参的明文直方图 GBDT 数学等价（INNOVATIONS 创新点 2 / 4），所以先在代理上筛方法，选出的组合再用真实 SGB 复核（S5.M2）。

**数据**：公开数据集，**开发证据**，不代表汇丰真实效果，不得换算成命中率或收益。逐行矩阵与分数只在私有目录。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m1_method_screen.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components/method_screen.py",
              REPO_ROOT / "modules/m5_modeling/components/method_screen_data.py"]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("决策记录:", cfg["decision_record"], "| step:", cfg["step_id"], "|", cfg["purpose"])
print("seed:", cfg["seed"], "| seeds:", cfg["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
for c in COMPONENTS:
    print("组件:", c.relative_to(REPO_ROOT), "sha256:" + sha(c))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RAW = pathlib.Path(os.environ[cfg["raw_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
NAMES = cfg["datasets"]
R = {n: json.loads((RUN / (n + ".json")).read_text(encoding="utf-8")) for n in NAMES}
print("运行结果：%d / %d 个数据集" % (len(R), len(NAMES)))
assert set(R) == set(NAMES)

config: modules/m5_modeling/configs/s5_m1_method_screen.yaml sha256:761bae66da90
决策记录: registry/decision_records/DR-M5-012.yaml | step: S5.M1 | 文献优化方法（I2 / I3 / I6 / M1 / M2 / M4 / M5 / M6）在 14 个公开数据集任务上的增益、组合与缺口
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: 75c3bb8（工作区含本步未提交改动）
组件: modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
组件: modules/m5_modeling/components/method_screen_data.py sha256:e4bfdf6fe3d4
运行结果：14 / 14 个数据集


## 1. 运行来源与仓库组件复现

整批运行用的是会话临时目录中的筛选代码（21:05:40 冻结并记录 sha256，见下方打印），本步把它整理为仓库组件 `method_screen` / `method_screen_data`（参数移入配置、常量命名，计算不变）。下一格用仓库组件在 3 个数据集上从原始文件重新准备数据并重跑全部方法，逐项比较：私有矩阵逐位相同，所有方法的测试 AUROC、配对差区间与 p 值、选定配置与组合都必须完全相同。3 个数据集覆盖有 / 无非共同客户两条路径。

In [2]:
from modules.m5_modeling.components import method_screen as MS
from modules.m5_modeling.components import method_screen_data as MD

prep_sha, prep_name, screen_sha, screen_name, launched_at = (RUN / "fingerprint_launch.txt").read_text(
    encoding="utf-8").split()
print("运行代码启动指纹（%s）：%s %s… / %s %s…" % (launched_at, prep_name, prep_sha[:HASH_PREFIX_LEN], screen_name,
                                           screen_sha[:HASH_PREFIX_LEN]))
tmp = pathlib.Path(tempfile.mkdtemp())
rows = []
for n in cfg["repro_check"]:
    info = MD.build(n, RAW, tmp, cfg["prep"])
    a, b = np.load(PRIV / (n + ".npz")), np.load(tmp / (n + ".npz"))
    same_data = sorted(a.files) == sorted(b.files) and all(np.array_equal(a[k], b[k]) for k in a.files)
    rep = MS.Screen(n, cfg, tmp, tmp / "out").run()
    ref = R[n]
    d_auc = max(abs(rep["test"][k]["auroc"] - ref["test"][k]["auroc"]) for k in ref["test"])
    d_ci = max(abs(x[f] - y[f]) for x, y in zip(rep["contrasts"], ref["contrasts"]) for f in ("ci_low", "ci_high", "p"))
    same_sel = rep["configs"] == ref["configs"] and rep["val"]["combo_choice"] == ref["val"]["combo_choice"]
    rows.append({"dataset": n, "矩阵逐位相同": same_data, "方法数": len(ref["test"]), "测试 AUROC 最大差": d_auc,
                 "区间与 p 最大差": d_ci, "配置与选择相同": same_sel})
rep_df = pd.DataFrame(rows)
print("仓库组件复现运行结果：")
print(rep_df.to_string(index=False))
assert rep_df["矩阵逐位相同"].all() and rep_df["配置与选择相同"].all()
assert (rep_df["测试 AUROC 最大差"] == 0).all() and (rep_df["区间与 p 最大差"] == 0).all()

运行代码启动指纹（2026-11-02T21:05:40+08:00）：prep.py 221bc073e371… / screen.py 56929574cc55…


[05:36:54] coil2000 · tune_guest_central 用时 8.7s


[05:36:56] coil2000 · tune_m4 用时 1.9s


[05:37:00] coil2000 · tune_m1 用时 3.8s


[05:37:02] coil2000 · val_selection 用时 2.3s


[05:37:02] coil2000 · 组合候选（验证 AUC）{'P': 0.8045, 'P_M4': 0.8045, 'M1': 0.8079, 'P+M6': 0.7949, 'M1+M6': 0.7961, 'P_M4+M6': 0.7949} → 选 M1；融合 w=1.0；蒸馏 α=1.0


... (107 outputs left out)

[05:40:55] uci_bank · test_m5 用时 3.3s


[05:41:05] uci_bank · bootstrap 用时 10.6s


[05:41:05] uci_bank · 完成


仓库组件复现运行结果：
       dataset  矩阵逐位相同  方法数  测试 AUROC 最大差  区间与 p 最大差  配置与选择相同
      coil2000    True   25           0.0        0.0     True
mock_overlap20    True   28           0.0        0.0     True
      uci_bank    True   25           0.0        0.0     True


## 2. 数据

14 个任务的训练 / 测试规模、正例数与两方列数（只含聚合计数）。验证集正例数决定选择的稳定性：COIL2000、mock 重合 20、Orange 偏好的验证正例少于 200，选择噪声大。

In [3]:
meta = pd.DataFrame([{"数据集": n, "训练行": R[n]["meta"]["final_fit_rows"], "训练正例": R[n]["meta"]["final_fit_pos"],
                      "测试行": R[n]["meta"]["final_eval_rows"], "测试正例": R[n]["meta"]["final_eval_pos"],
                      "验证正例": R[n]["meta"]["tune_eval_pos"], "内地列": R[n]["meta"]["n_guest"],
                      "香港列": R[n]["meta"]["n_host"], "非共同客户行": R[n]["meta"]["extra_rows"],
                      "测试簇": R[n]["meta"]["cluster"]} for n in NAMES])
print(meta.to_string(index=False))
for n in NAMES:
    print("·", n, "|", R[n]["meta"]["source"], "|", R[n]["meta"]["split"])

             数据集    训练行  训练正例    测试行  测试正例  验证正例  内地列  香港列  非共同客户行 测试簇
    santander_g3 135819  4955  20351   586  1215   19   25       0  客户
   mock_balanced  78267  2892  21735   645   765   19   25       0  客户
  mock_overlap20  15555   596   4370   130   157   20   25   62712  客户
  mock_overlap50  38925  1432  10884   316   381   20   25   39342  客户
  mock_overlap80  62564  2273  17391   511   603   20   25   15703  客户
        uci_bank  32950  3712   8238   928   742   12    7       0   行
    orange_churn  40000  2938  10000   734   588   80   67       0   行
   orange_upsell  40000  2946  10000   736   589   80   67       0   行
orange_appetency  40000   712  10000   178   142   80   67       0   行
          fedads 300000  1778 200000  1253   325   22   23  312204   行
      homecredit  80000  6408  20000  1602  1282  151   17       0   行
    santander_tx  40000  4096  10000  1024   819  100  100       0   行
      ieee_part0  28846  1112   7212   174   257  357   46       0   行
      

## 3. 只在验证集上做出的选择

每个数据集：内地单边与集中式各自 48 组网格调参（细分桶 33、先验初始分数），M4 在集中式前 8 组上加行采样 0.5 / 0.7，M1 在内地交叉拟合 logit 上调残差模型。组合候选 {P = 优化版、P_M4、M1、M1x 及各自 + M6 bagging} 按验证 AUROC 选，差距 ≤ 0.001 时取计算量最小者。融合权重 w 与蒸馏系数 α 同样只看验证集。

In [4]:
KEYS = ("lr", "depth", "lam", "col", "sub", "rounds")
sel = []
for n in NAMES:
    c, v = R[n]["configs"], R[n]["val"]
    cand = {k: x["val_auc"] for k, x in v["combo_candidates"].items()}
    sel.append({"数据集": n, "内地单边": "/".join(str(c["guest"][k]) for k in KEYS),
                "集中式（优化版联邦）": "/".join(str(c["central"][k]) for k in KEYS),
                "M4": "/".join(str(c["m4"][k]) for k in KEYS), "M1 残差": "/".join(str(c["m1"][k]) for k in KEYS),
                "验证最优候选": max(cand, key=cand.get), "选定组合": v["combo_choice"],
                "融合 w": v["fuse"]["w"], "蒸馏 α": v["student"]["alpha"]})
print("选定配置（学习率/深度/λ/列采样/行采样/轮数）与验证集上的选择：")
print(pd.DataFrame(sel).to_string(index=False))
val = pd.DataFrame({n: {k: x["val_auc"] for k, x in R[n]["val"]["combo_candidates"].items()} for n in NAMES}).T
print("组合候选的验证 AUROC：")
print(val.round(cfg["round_auc"]).to_string())

选定配置（学习率/深度/λ/列采样/行采样/轮数）与验证集上的选择：
             数据集                   内地单边             集中式（优化版联邦）                     M4                  M1 残差 验证最优候选  选定组合  融合 w  蒸馏 α
    santander_g3   0.1/4/0.1/0.7/1.0/23   0.1/6/0.1/0.7/1.0/33   0.1/6/0.1/0.7/1.0/33   0.1/6/0.1/1.0/1.0/41     M1    M1   1.0  0.75
   mock_balanced  0.3/6/0.1/0.7/1.0/195   0.3/6/1.0/1.0/1.0/88   0.3/6/1.0/1.0/1.0/88   0.3/6/0.1/1.0/1.0/47   P+M6  P+M6   0.8  0.50
  mock_overlap20   0.3/6/0.1/0.7/1.0/33   0.3/5/0.1/0.7/1.0/52   0.3/6/1.0/0.7/0.7/71   0.3/4/1.0/0.7/1.0/44   P_M4  P_M4   0.7  1.00
  mock_overlap50 0.3/6/10.0/0.7/1.0/120   0.3/6/1.0/0.7/1.0/62   0.1/6/0.1/1.0/0.7/90   0.3/6/0.1/0.7/1.0/37   P_M4  P_M4   0.8  0.50
  mock_overlap80  0.1/6/0.1/0.7/1.0/236   0.3/6/1.0/1.0/1.0/83  0.1/6/0.1/1.0/0.7/114  0.3/6/10.0/0.7/1.0/76   P+M6  P+M6   0.8  1.00
        uci_bank  0.3/5/10.0/0.7/1.0/15   0.3/4/1.0/0.7/1.0/34   0.3/4/1.0/0.7/1.0/34    0.3/5/1.0/0.7/1.0/2      P    M1   0.7  1.00
    orange_churn   0.1/5/0.

## 读法

- **来源**：仓库组件在 COIL2000、mock 重合 20、UCI 上从原始文件重新准备数据并重跑全部方法，私有矩阵逐位相同，所有方法的测试 AUROC、区间、p 值、选定配置与组合完全相同（最大差 0）。所以整批结果可视为由仓库组件产生。
- **数据**：14 个任务的测试正例 130 ~ 1,602；验证正例 70 ~ 1,282。FedAds 与 mock 重合三档带非共同客户的内地行（1.6 万 ~ 31 万）。
- **选择**：验证集最优候选在 8 / 14 个任务上与 M1 并列或就是 M1。因为并列时取计算量最小者，而 M1 的联邦残差模型很小（UCI 只有 2 轮），组合规则在这 8 个任务上都选了含 M1 的组合；测试结果见 S5.M1-2。
- **非共同客户**：在 FedAds 上，验证集认为 M1x 与 M6 都比优化版差，测试集上二者都有增益。单一验证集分不开 0.005 量级的差距，下一轮改为训练集交叉验证选择（DR-M5-012 复核记录）。